In [1]:
# Standard Library
from pathlib import Path
import pickle

# LangChain
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

C:\Users\mohme\AppData\Local\Temp\ipykernel_30972\1003665118.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
d:\career-ai-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = Path("../data")
OUTPUT_PATH = Path("../storage/processed")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
loader = DirectoryLoader(
    path=str(DATA_PATH),
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    use_multithreading=True,
)

In [5]:
from pathlib import Path

print(Path.cwd())

d:\career-ai-agent\data\interview


In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent

print(PROJECT_ROOT)

d:\career-ai-agent


In [7]:
DATA_PATH = PROJECT_ROOT / "data"
OUTPUT_PATH = PROJECT_ROOT / "storage" / "processed"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(DATA_PATH)

d:\career-ai-agent\data


In [8]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project Root:", PROJECT_ROOT)

Project Root: d:\career-ai-agent


In [9]:
loader = DirectoryLoader(
    path=str(DATA_PATH),
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    use_multithreading=True,
)

In [10]:
documents = loader.load()

print(f"Loaded {len(documents)} pages")

  0%|          | 0/12 [00:00<?, ?it/s]Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
100%|██████████| 12/12 [00:07<00:00,  1.65it/s]

Loaded 90 pages


In [11]:
documents[0].metadata

{'producer': 'Microsoft® Word for Microsoft 365',
 'creator': 'Microsoft® Word for Microsoft 365',
 'creationdate': '2024-05-22T15:58:46-04:00',
 'title': 'Consultant Handout Templates (Under-Grad-Combo) v2',
 'author': 'Allison Viverette',
 'moddate': '2024-05-22T15:58:46-04:00',
 'source': 'd:\\career-ai-agent\\data\\interview\\CMU_Behavioral_Interview_Guide.pdf',
 'total_pages': 3,
 'page': 0,
 'page_label': '1'}

In [12]:
documents[0].page_content[:500]

'Behavioral Interview Guide  \nOverview: \nThroughout an interview process, you will likely have one or more behavioral interviews \nwith a company of interest. If you are ever unsure of the type of interview you are \nscheduled for, you can ask your recruiter for clarity in advance. Behavioral Interviews \nallow recruiters and members of a hiring team to assess how and if your past \nexperiences, behaviors, and skills demonstrate the key characteristics and competencies \nthey have deemed essential for'

In [13]:
from pathlib import Path

for doc in documents:
    source = Path(doc.metadata["source"])

    doc.metadata["filename"] = source.name
    doc.metadata["category"] = source.parent.name

In [14]:
documents[0].metadata

{'producer': 'Microsoft® Word for Microsoft 365',
 'creator': 'Microsoft® Word for Microsoft 365',
 'creationdate': '2024-05-22T15:58:46-04:00',
 'title': 'Consultant Handout Templates (Under-Grad-Combo) v2',
 'author': 'Allison Viverette',
 'moddate': '2024-05-22T15:58:46-04:00',
 'source': 'd:\\career-ai-agent\\data\\interview\\CMU_Behavioral_Interview_Guide.pdf',
 'total_pages': 3,
 'page': 0,
 'page_label': '1',
 'filename': 'CMU_Behavioral_Interview_Guide.pdf',
 'category': 'interview'}

In [15]:
import pickle

with open(OUTPUT_PATH / "documents.pkl", "wb") as f:
    pickle.dump(documents, f)

print("Documents Saved ✅")

Documents Saved ✅


In [16]:
with open(OUTPUT_PATH / "documents.pkl", "rb") as f:
    loaded_documents = pickle.load(f)

print(len(loaded_documents))
print(loaded_documents[0].metadata)

90
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-05-22T15:58:46-04:00', 'title': 'Consultant Handout Templates (Under-Grad-Combo) v2', 'author': 'Allison Viverette', 'moddate': '2024-05-22T15:58:46-04:00', 'source': 'd:\\career-ai-agent\\data\\interview\\CMU_Behavioral_Interview_Guide.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'filename': 'CMU_Behavioral_Interview_Guide.pdf', 'category': 'interview'}


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

In [19]:
chunks = text_splitter.split_documents(documents)

print(len(chunks))

285


In [20]:
chunks[0].page_content

'Behavioral Interview Guide  \nOverview: \nThroughout an interview process, you will likely have one or more behavioral interviews \nwith a company of interest. If you are ever unsure of the type of interview you are \nscheduled for, you can ask your recruiter for clarity in advance. Behavioral Interviews \nallow recruiters and members of a hiring team to assess how and if your past \nexperiences, behaviors, and skills demonstrate the key characteristics and competencies \nthey have deemed essential for the role for which you are interviewing and/or the \ncompany. They also allow the hiring team to delve further into the experiences listed \nthroughout your resume, and in some cases, learn more about you outside of what you \nhave included on that document.  \n \nBefore the Interview:  \n• Spend time thoroughly researching the company. You will likely be asked why you are \ninterested in the company, and you want to be able to connect your interests back'

In [21]:
import pickle

chunks_path = OUTPUT_PATH / "chunks.pkl"

with open(chunks_path, "wb") as f:
    pickle.dump(chunks, f)

print(f"✅ Saved {len(chunks)} chunks")
print(f"📁 {chunks_path}")

✅ Saved 285 chunks
📁 d:\career-ai-agent\storage\processed\chunks.pkl


In [22]:
import json
from datetime import datetime

chunk_config = {
    "splitter": "RecursiveCharacterTextSplitter",
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "total_chunks": len(chunks),
    "created_at": datetime.now().isoformat()
}

with open(OUTPUT_PATH / "chunk_config.json", "w", encoding="utf-8") as f:
    json.dump(chunk_config, f, indent=4)

print("✅ Chunk config saved")

✅ Chunk config saved


In [23]:
from langchain_huggingface import HuggingFaceEmbeddings


In [24]:
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3726.62it/s]


In [25]:
vector = embedding_model.embed_query(
    "What is a behavioral interview?"
)

print(type(vector))
print(len(vector))

print(vector[:10])

<class 'list'>
384
[-0.01770656742155552, 0.14938989281654358, -0.02326248027384281, -0.03612237423658371, 0.022301094606518745, 0.04534501954913139, 0.05258382111787796, -0.019599227234721184, -0.0017073913477361202, -0.023415060713887215]


In [28]:
from langchain_chroma import Chroma

In [29]:
VECTOR_DB_PATH = PROJECT_ROOT / "storage" / "vector_db"

VECTOR_DB_PATH.mkdir(parents=True, exist_ok=True)

print(VECTOR_DB_PATH)

d:\career-ai-agent\storage\vector_db


In [30]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=str(VECTOR_DB_PATH),
)

print("✅ Vector Database Created")

✅ Vector Database Created


In [31]:
collection = vector_db._collection

print(collection.count())

285


Result 1
{'moddate': '2024-05-22T15:58:46-04:00', 'author': 'Allison Viverette', 'page': 0, 'category': 'interview', 'creator': 'Microsoft® Word for Microsoft 365', 'page_label': '1', 'producer': 'Microsoft® Word for Microsoft 365', 'source': 'd:\\career-ai-agent\\data\\interview\\CMU_Behavioral_Interview_Guide.pdf', 'filename': 'CMU_Behavioral_Interview_Guide.pdf', 'title': 'Consultant Handout Templates (Under-Grad-Combo) v2', 'creationdate': '2024-05-22T15:58:46-04:00', 'total_pages': 3}
Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or more behavioral interviews 
with a company of interest. If you are ever unsure of the type of interview you are 
scheduled for, you can ask your recruiter for clarity in advance. Behavioral Interview
Result 2
{'source': 'd:\\career-ai-agent\\data\\interview\\CMU_Behavioral_Interview_Guide.pdf', 'author': 'Allison Viverette', 'creationdate': '2024-05-22T15:58:46-04:00', 'moddate': '2024-05-22T15:58:46-

In [34]:
def search_documents(query: str, k: int = 3):
    retriever = vector_db.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

    return retriever.invoke(query)

In [35]:
results = search_documents("behavioral interview")

In [36]:
for i, doc in enumerate(results, start=1):
    print("=" * 60)
    print(f"Result {i}")
    print(doc.metadata)
    print(doc.page_content[:300])

Result 1
{'producer': 'Microsoft® Word for Microsoft 365', 'page_label': '1', 'creator': 'Microsoft® Word for Microsoft 365', 'filename': 'CMU_Behavioral_Interview_Guide.pdf', 'author': 'Allison Viverette', 'category': 'interview', 'total_pages': 3, 'source': 'd:\\career-ai-agent\\data\\interview\\CMU_Behavioral_Interview_Guide.pdf', 'title': 'Consultant Handout Templates (Under-Grad-Combo) v2', 'moddate': '2024-05-22T15:58:46-04:00', 'page': 0, 'creationdate': '2024-05-22T15:58:46-04:00'}
Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or more behavioral interviews 
with a company of interest. If you are ever unsure of the type of interview you are 
scheduled for, you can ask your recruiter for clarity in advance. Behavioral Interview
Result 2
{'creationdate': '2024-05-22T15:58:46-04:00', 'title': 'Consultant Handout Templates (Under-Grad-Combo) v2', 'author': 'Allison Viverette', 'producer': 'Microsoft® Word for Microsoft 365', 'total

In [37]:
for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print(f"File     : {doc.metadata['filename']}")
    print(f"Category : {doc.metadata['category']}")
    print(f"Page     : {doc.metadata['page'] + 1}")

    print("\nContent:")
    print(doc.page_content[:300])

Result 1
File     : CMU_Behavioral_Interview_Guide.pdf
Category : interview
Page     : 1

Content:
Behavioral Interview Guide  
Overview: 
Throughout an interview process, you will likely have one or more behavioral interviews 
with a company of interest. If you are ever unsure of the type of interview you are 
scheduled for, you can ask your recruiter for clarity in advance. Behavioral Interview
Result 2
File     : CMU_Behavioral_Interview_Guide.pdf
Category : interview
Page     : 3

Content:
mentioned previously, I am currently getting my master’s in Computer Vision at Carnegie Mellon, with an 
interest in spherical CNNs. I am very interested in learning more about your work across the artificial 
intelligence industry. Would you have 15 minutes of time in the coming weeks to set up a p
Result 3
File     : Meta_ML_onsite_interview_prep.pdf
Category : interview
Page     : 5

Content:
several different algorithms and understanding the tradeoffs is helpful. For
example, be able to expla